# Cross-encoder 2 (XLM-RoBERTa) for Business Entity Resolution (Ekdant.exe)
1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Click the **folder icon** on the left, then the **upload** icon, and upload the SAME 12 files as last time: `xtrain_part00..03.parquet`, `xvt_part00.parquet`, `xtest_part00..06.parquet` (from `amazon-ml/colab_gpu`).
3. **Runtime → Run all.** (Optional: while it trains, also upload the `xextra_part*.parquet` files from `amazon-ml/colab_gpu2` — they get scored at the end.) It takes about 60–80 minutes. At the end, `xenc2_scores.csv.gz` downloads automatically. Put it in the `amazon-ml` folder.

Model: `xlm-roberta-base` (MIT license, 278M parameters), fine-tuned as a record/entity pair classifier, 1 epoch.


In [ ]:
import glob, os, time, math, numpy as np, pandas as pd, torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
n = {k: len(glob.glob(f'/content/{k}_part*.parquet')) for k in ('xtrain', 'xvt', 'xtest')}
print(n)
assert n == {'xtrain': 4, 'xvt': 1, 'xtest': 7}, 'upload all 12 files first (folder icon on the left)'


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
torch.manual_seed(1); np.random.seed(1)
MODEL = 'xlm-roberta-base'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1).cuda()
rd = lambda k: pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(f'/content/{k}_part*.parquet'))], ignore_index=True)
tr = rd('xtrain'); print('train pairs', len(tr), 'positive share', tr.label.mean())
def enc(a, b):
    x = tok(list(a), list(b), truncation=True, max_length=128, padding=True, return_tensors='pt')
    return {k: v.cuda(non_blocking=True) for k, v in x.items()}


In [ ]:
BS, EPOCHS, LR = 64, 1, 2e-5
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * math.ceil(len(tr) / BS)
sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
scaler = torch.cuda.amp.GradScaler()
lossf = torch.nn.BCEWithLogitsLoss()
model.train()
t0 = time.time()
for ep in range(EPOCHS):
    idx = np.random.RandomState(10 + ep).permutation(len(tr))
    nb = math.ceil(len(idx) / BS)
    for s, i in enumerate(range(0, len(idx), BS)):
        b = tr.iloc[idx[i:i + BS]]
        x = enc(b.a, b.b)
        y = torch.tensor(b.label.values, dtype=torch.float32).cuda()
        with torch.autocast('cuda', dtype=torch.float16):
            out = model(**x).logits.squeeze(-1)
        loss = lossf(out.float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sch.step()
        if s % 500 == 0:
            el = time.time() - t0
            print(f'epoch {ep} step {s}/{nb} loss {loss.item():.4f} {el:.0f}s, about {el / max(s, 1) * (nb - s) / 60:.0f} min left in this epoch', flush=True)
print('training done', round(time.time() - t0), 's')


In [ ]:
model.eval()
def predict(df, bs=512):
    order = np.argsort((df.a.str.len() + df.b.str.len()).values)
    out = np.empty(len(df), dtype=np.float32)
    with torch.no_grad():
        for i in range(0, len(order), bs):
            j = order[i:i + bs]; b = df.iloc[j]
            with torch.autocast('cuda', dtype=torch.float16):
                out[j] = model(**enc(b.a, b.b)).logits.squeeze(-1).float().cpu().numpy()
    return out
vt = rd('xvt'); vt['score'] = predict(vt)
from sklearn.metrics import roc_auc_score
print('hold-out pairs', len(vt), 'AUC', round(roc_auc_score(vt.label, vt.score), 4))
te = rd('xtest'); t0 = time.time(); te['score'] = predict(te); print('test pairs', len(te), round(time.time() - t0), 's')
res = pd.concat([vt[['rid', 's1', 'score']].assign(split='vt'), te[['rid', 's1', 'score']].assign(split='test')], ignore_index=True)
res.to_csv('/content/xenc2_scores.csv.gz', index=False)
print('written', len(res), os.path.getsize('/content/xenc2_scores.csv.gz') // 1_000_000, 'MB')


In [ ]:
from google.colab import files
files.download('/content/xenc2_scores.csv.gz')   # put this file in your amazon-ml folder (if the download does not start: right-click the file in the Files panel -> Download)


In [ ]:
# OPTIONAL extra step: if you uploaded xextra_part*.parquet files (from amazon-ml/colab_gpu2), they are scored here.
ex = sorted(glob.glob('/content/xextra_part*.parquet'))
print('extra files', len(ex))
if ex:
    xe = pd.concat([pd.read_parquet(f) for f in ex], ignore_index=True)
    t0 = time.time(); xe['score'] = predict(xe); print('extra pairs', len(xe), round(time.time() - t0), 's')
    xe[['rid', 's1', 'score', 'split']].to_csv('/content/xenc2_extra_scores.csv.gz', index=False)
    files.download('/content/xenc2_extra_scores.csv.gz')   # put this file in your amazon-ml folder too
